In [ ]:
# =============================================
# SECTION 1 — 3.2 Part 1: PCA with Mean-Filling
# =============================================

import os
import numpy as np
import pandas as pd

# --- Path  ---
from pathlib import Path

PROJECT_ROOT = Path.cwd()

def resolve_existing(*candidates: str) -> str:
    for c in candidates:
        p = (PROJECT_ROOT / c)
        if p.exists():
            return str(p)
    tried = "\n".join([f"- {PROJECT_ROOT / c}" for c in candidates])
    raise FileNotFoundError(f"Could not find required file. Tried:\n{tried}")

# Common dataset candidates 
RATINGS_PATH = resolve_existing(
     r"C:\ml-20m\ml-20m\ratings.csv",)


# ----- Paths -----
OUT_DIR = "SECTION1_DimensionalityReduction/data"

# ----- Load saved outputs from Section 3.1 -----
ratings = pd.read_csv(RATINGS_PATH)
targets_items = pd.read_csv(os.path.join(OUT_DIR, "target_items.csv"))

I1 = int(targets_items.loc[targets_items["item_label"] == "I1_low_popularity", "movieId"].values[0])
I2 = int(targets_items.loc[targets_items["item_label"] == "I2_high_popularity", "movieId"].values[0])

target_items = [I1, I2]

# =============================================
# Step 1: Create user–item rating matrix
# Rows = users, Columns = target items (I1, I2)
# =============================================
rating_matrix = ratings[ratings["movieId"].isin(target_items)] \
    .pivot(index="userId", columns="movieId", values="rating")

print("Initial rating matrix (with NaNs):")
display(rating_matrix.head())

# =============================================
# Step 2: Compute average rating for each target item
# =============================================
item_means = rating_matrix.mean(axis=0).round(2)

print("\nAverage rating for target items:")
for item_id, mean_val in item_means.items():
    print(f"Item {item_id}: mean rating = {mean_val:.2f}")

# Save means
item_means.to_csv(os.path.join(OUT_DIR, "target_item_means.csv"))

# =============================================
# Step 3: Mean-filling missing ratings
# Replace NaN with corresponding item mean
# =============================================
rating_matrix_filled = rating_matrix.copy()

for item_id in rating_matrix_filled.columns:
    rating_matrix_filled[item_id] = rating_matrix_filled[item_id].fillna(item_means[item_id])

rating_matrix_filled = rating_matrix_filled.round(2)

print("\nRating matrix after mean-filling:")
display(rating_matrix_filled.head())

# Save filled matrix
rating_matrix_filled.to_csv(os.path.join(OUT_DIR, "rating_matrix_mean_filled.csv"))



Initial rating matrix (with NaNs):


movieId,235,118758
userId,,
5,3.0,NaN
20,3.5,NaN
21,3.0,NaN
23,3.0,NaN
27,3.0,NaN



Average rating for target items:
Item 235: mean rating = 3.66
Item 118758: mean rating = 1.50

Rating matrix after mean-filling:


movieId,235,118758
userId,,
5,3.0,1.5
20,3.5,1.5
21,3.0,1.5
23,3.0,1.5
27,3.0,1.5


In [ ]:
# =============================================
# 3.2 Part 1 — Step 3 & Step 4
# Step 3: Average rating for each target item
# Step 4: Difference = rating - item_mean (mean-centering)
# =============================================

# Step 3: compute mean rating for each item (from original matrix with NaNs)
item_means = rating_matrix.mean(axis=0).round(2)

print("\n[Step 3] Average rating for each target item:")
for item_id, mean_val in item_means.items():
    print(f"Item {item_id}: mean = {mean_val:.2f}")

# Step 4: compute differences (rating - mean) for each item
# (This keeps NaNs where rating is missing )
diff_matrix = rating_matrix.sub(item_means, axis=1).round(2)

print("\n[Step 4] Difference matrix (rating - item mean):")
display(diff_matrix.head())

# Save outputs
item_means.to_csv(os.path.join(OUT_DIR, "target_item_means.csv"))
diff_matrix.to_csv(os.path.join(OUT_DIR, "diff_matrix_item_mean_centered.csv"))




[Step 3] Average rating for each target item:
Item 235: mean = 3.66
Item 118758: mean = 1.50

[Step 4] Difference matrix (rating - item mean):


movieId,235,118758
userId,,
5,-0.66,NaN
20,-0.16,NaN
21,-0.66,NaN
23,-0.66,NaN
27,-0.66,NaN


In [3]:
# =============================================
# 3.2 Part 1 — Step 5, Step 6, Step 7
# Step 5: Compute covariance for each two items
# Step 6: Generate covariance matrix
# Step 7: Determine top-5 and top-10 peers using covariance matrix
# =============================================

import numpy as np
import pandas as pd

# ----- Step 5: Mean-center (after mean-filling) -----
centered_matrix = rating_matrix_filled.sub(item_means, axis=1).round(2)

print("\nMean-centered matrix:")
display(centered_matrix.head())

# ----- Step 6: Covariance matrix -----
cov_matrix = np.cov(centered_matrix.T, bias=False)
cov_matrix = np.round(cov_matrix, 2)

cov_df = pd.DataFrame(
    cov_matrix,
    index=centered_matrix.columns,
    columns=centered_matrix.columns
)

print("\nCovariance Matrix:")
display(cov_df)

# Save covariance matrix
cov_df.to_csv(os.path.join(OUT_DIR, "covariance_matrix_items.csv"))

# ----- Step 7: Top-5 and Top-10 peers for each target item -----
peers_results = {}

for item in cov_df.columns:
    # Remove self-covariance
    peers = cov_df[item].drop(item)

    # Sort by absolute covariance 
    peers_sorted = peers.reindex(peers.abs().sort_values(ascending=False).index)

    top_5 = peers_sorted.head(5)
    top_10 = peers_sorted.head(10)

    peers_results[item] = {
        "top_5_peers": top_5,
        "top_10_peers": top_10
    }

    print(f"\nItem {item} — Top Peers (by covariance):")
    print("Top-5 peers:")
    display(top_5.to_frame(name="covariance"))

    print("Top-10 peers:")
    display(top_10.to_frame(name="covariance"))

# Save peers info
with open(os.path.join(OUT_DIR, "item_peers_covariance.txt"), "w") as f:
    for item, data in peers_results.items():
        f.write(f"Item {item}\n")
        f.write("Top-5 peers:\n")
        f.write(data["top_5_peers"].to_string())
        f.write("\n\nTop-10 peers:\n")
        f.write(data["top_10_peers"].to_string())
        f.write("\n\n")




Mean-centered matrix:


movieId,235,118758
userId,,
5,-0.66,0.0
20,-0.16,0.0
21,-0.66,0.0
23,-0.66,0.0
27,-0.66,0.0



Covariance Matrix:


movieId,235,118758
movieId,,
235,0.92,0.0
118758,0.00,0.0



Item 235 — Top Peers (by covariance):
Top-5 peers:


...

,covariance
movieId,
118758,0.0



Item 118758 — Top Peers (by covariance):
Top-5 peers:


,covariance
movieId,
235,0.0


Top-10 peers:


,covariance
movieId,
235,0.0


In [4]:
# =============================================
# 3.2 Part 1 — Step 8, 9, 10, 11
# Step 8: Reduced space per user using top-5 peers
# Step 9: Predict missing ratings for I1 & I2 using top-5 peers
# Step 10: Reduced space per user using top-10 peers
# Step 11: Predict missing ratings for I1 & I2 using top-10 peers
# =============================================

import numpy as np
import pandas as pd

# --------get peers list from covariance matrix --------
def get_top_peers(cov_df: pd.DataFrame, item_id: int, top_n: int) -> list:
    peers = cov_df[item_id].drop(item_id)
    # sort by absolute covariance strength
    peers_sorted = peers.reindex(peers.abs().sort_values(ascending=False).index)
    return peers_sorted.head(top_n).index.tolist()

# --------reduced representation and prediction --------
def reduced_space_and_predict(
    rating_matrix: pd.DataFrame,
    rating_matrix_filled: pd.DataFrame,
    item_means: pd.Series,
    cov_df: pd.DataFrame,
    top_n: int
):
    """
    For each user:
    - Reduced space vector = mean-centered ratings of the selected peers
    For each missing rating in original matrix:
    - Predict using covariance-weighted combination of peer deviations:
        r_hat(u,i) = mean(i) + sum_j cov(i,j) * (r(u,j)-mean(j)) / sum_j |cov(i,j)|
    """
    # mean-centered filled matrix (no NaNs)
    centered = rating_matrix_filled.sub(item_means, axis=1).round(2)

    # store reduced space per user (for each target item, reduced vector over peers)
    reduced_spaces = {}

    # store predictions for originally missing entries only
    preds = []

    items = list(rating_matrix.columns)

    for item in items:
        peers = get_top_peers(cov_df, item, top_n=top_n)

        # Reduced space for each user = centered ratings of peers
        reduced_spaces[item] = centered[peers].copy()
        reduced_spaces[item].columns = [f"peer_{p}" for p in peers]

        # Predict ONLY where original rating is missing (NaN)
        missing_users = rating_matrix.index[rating_matrix[item].isna()].tolist()

        for u in missing_users:
            numerator = 0.0
            denom = 0.0

            for p in peers:
                w = float(cov_df.loc[item, p])            # covariance weight
                dev = float(centered.loc[u, p])           # (r(u,p) - mean(p))
                numerator += w * dev
                denom += abs(w)

            # if denom is zero fallback to item mean
            pred = float(item_means[item]) if denom == 0 else float(item_means[item] + (numerator / denom))

            preds.append({
                "userId": u,
                "itemId": item,
                "peer_mode": f"top_{top_n}",
                "predicted_rating": round(pred, 2)
            })

    # Combine reduced spaces into a single dict of DataFrames
    return reduced_spaces, pd.DataFrame(preds)


# =========================
# Step 8 + Step 9 (Top-5)
# =========================
reduced_5, preds_5 = reduced_space_and_predict(
    rating_matrix=rating_matrix,
    rating_matrix_filled=rating_matrix_filled,
    item_means=item_means,
    cov_df=cov_df,
    top_n=5
)

print("\n[Step 8] Reduced dimensional space (Top-5 peers):")
for item, df in reduced_5.items():
    print(f"Item {item} reduced space (first rows):")
    display(df.head())

print("\n[Step 9] Predictions for ORIGINAL missing ratings (Top-5 peers):")
display(preds_5.head())

# Save
for item, df in reduced_5.items():
    df.round(2).to_csv(os.path.join(OUT_DIR, f"reduced_space_top5_item_{item}.csv"))
preds_5.to_csv(os.path.join(OUT_DIR, "predictions_top5.csv"), index=False)


# ==========================
# Step 10 + Step 11 (Top-10)
# ==========================
reduced_10, preds_10 = reduced_space_and_predict(
    rating_matrix=rating_matrix,
    rating_matrix_filled=rating_matrix_filled,
    item_means=item_means,
    cov_df=cov_df,
    top_n=10
)

print("\n[Step 10] Reduced dimensional space (Top-10 peers):")
for item, df in reduced_10.items():
    print(f"Item {item} reduced space (first rows):")
    display(df.head())

print("\n[Step 11] Predictions for ORIGINAL missing ratings (Top-10 peers):")
display(preds_10.head())

# Save
for item, df in reduced_10.items():
    df.round(2).to_csv(os.path.join(OUT_DIR, f"reduced_space_top10_item_{item}.csv"))
preds_10.to_csv(os.path.join(OUT_DIR, "predictions_top10.csv"), index=False)




[Step 8] Reduced dimensional space (Top-5 peers):
Item 235 reduced space (first rows):


,peer_118758
userId,
5,0.0
20,0.0
21,0.0
23,0.0
27,0.0


Item 118758 reduced space (first rows):


,peer_235
userId,
5,-0.66
20,-0.16
21,-0.66
23,-0.66
27,-0.66



[Step 9] Predictions for ORIGINAL missing ratings (Top-5 peers):


...

,peer_118758
userId,
5,0.0
20,0.0
21,0.0
23,0.0
27,0.0


Item 118758 reduced space (first rows):


,peer_235
userId,
5,-0.66
20,-0.16
21,-0.66
23,-0.66
27,-0.66



[Step 11] Predictions for ORIGINAL missing ratings (Top-10 peers):


,userId,itemId,peer_mode,predicted_rating
0,5,118758,top_10,1.5
1,20,118758,top_10,1.5
2,21,118758,top_10,1.5
3,23,118758,top_10,1.5
4,27,118758,top_10,1.5


In [5]:
# =============================================
# 3.2 Part 1 — Step 12
# Compare predictions from Step 9 (top-5) vs Step 11 (top-10)
# =============================================

import pandas as pd
import numpy as np

# Ensure the prediction columns are consistent
p5 = preds_5.copy()
p10 = preds_10.copy()

# Rename for clarity
p5 = p5.rename(columns={"predicted_rating": "pred_top5"})
p10 = p10.rename(columns={"predicted_rating": "pred_top10"})

# Merge on userId + itemId (only missing entries were predicted in both)
compare_df = pd.merge(
    p5[["userId", "itemId", "pred_top5"]],
    p10[["userId", "itemId", "pred_top10"]],
    on=["userId", "itemId"],
    how="outer"
)

# Compute difference
compare_df["abs_diff"] = (compare_df["pred_top5"] - compare_df["pred_top10"]).abs().round(2)
compare_df["same_prediction"] = compare_df["abs_diff"].eq(0)

print("\n[Step 12] Top-5 vs Top-10 Prediction Comparison:")
display(compare_df.head(20))

# Summary stats
max_diff = float(compare_df["abs_diff"].max()) if len(compare_df) else 0.0
mean_diff = float(compare_df["abs_diff"].mean()) if len(compare_df) else 0.0
same_pct = float(compare_df["same_prediction"].mean() * 100) if len(compare_df) else 0.0

print("\nSummary:")
print(f"Mean absolute difference: {mean_diff:.2f}")
print(f"Max absolute difference:  {max_diff:.2f}")
print(f"% identical predictions:  {same_pct:.2f}%")

# Save comparison
compare_df.to_csv(os.path.join(OUT_DIR, "compare_top5_vs_top10.csv"), index=False)

# -----------------------
# Report-ready comment
# -----------------------
comment_lines = []
comment_lines.append("Comment (Step 12):")
comment_lines.append(
    "In Part 1, only two target items (I1 and I2) are considered. "
    "Therefore, each item has only one available peer (the other item). "
    "As a result, the top-5 and top-10 peer sets become identical, "
    "leading to the same reduced representation and the same predicted ratings."
)

# If somehow differences appear (edge case), add note
if max_diff > 0:
    comment_lines.append(
        "Small differences may appear due to numerical rounding or if additional peers "
        "were included in the peer set. However, the overall behavior remains consistent."
    )

print("\n" + "\n".join(comment_lines))


[Step 12] Top-5 vs Top-10 Prediction Comparison:


,userId,itemId,pred_top5,pred_top10,abs_diff,same_prediction
0,5,118758,1.5,1.5,0.0,True
1,20,118758,1.5,1.5,0.0,True
2,21,118758,1.5,1.5,0.0,True
3,23,118758,1.5,1.5,0.0,True
4,27,118758,1.5,1.5,0.0,True
5,29,118758,1.5,1.5,0.0,True
6,32,118758,1.5,1.5,0.0,True
7,46,118758,1.5,1.5,0.0,True
8,54,118758,1.5,1.5,0.0,True
9,58,118758,1.5,1.5,0.0,True



Summary:
Mean absolute difference: 0.00
Max absolute difference:  0.00
% identical predictions:  100.00%

Comment (Step 12):
In Part 1, only two target items (I1 and I2) are considered. Therefore, each item has only one available peer (the other item). As a result, the top-5 and top-10 peer sets become identical, leading to the same reduced representation and the same predicted ratings.
